# ARIA — Adaptive Reasoning Intelligence for Analytics
## ABB EngineeredX 2.0 | Problem Statement #8
### SRM University AP | Bhagavathi Neha Chinnam

This notebook walks through the full ARIA pipeline on the Telco Customer Churn dataset.

**Pipeline stages:**
1. Load and profile the dataset
2. Classify the ML task (with Devil's Advocate check)
3. Run AutoML and compare models
4. Compute feature importance
5. Generate tri-persona report (Engineer / Manager / Regulator)

In [ ]:
# Install dependencies (run once)
# !pip install pycaret plotly scikit-learn lightgbm xgboost pandas numpy

In [ ]:
import sys, os
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from utils.aria_pipeline import (
    profile_dataset,
    classify_task,
    run_automl,
    get_feature_importance,
    generate_report
)

print('ARIA pipeline loaded.')

## Step 1 — Load Dataset
Using the Telco Customer Churn dataset (available on Kaggle).

In [ ]:
# Load dataset — replace with your own CSV path if needed
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

# For demo: create a small synthetic dataset
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'tenure':           np.random.randint(1, 72, n),
    'MonthlyCharges':   np.random.uniform(20, 120, n),
    'TotalCharges':     np.random.uniform(100, 8000, n),
    'Contract':         np.random.choice(['Month-to-month', 'One year', 'Two year'], n),
    'PaymentMethod':    np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
    'InternetService':  np.random.choice(['DSL', 'Fiber optic', 'No'], n),
    'TechSupport':      np.random.choice(['Yes', 'No', 'No internet service'], n),
    'SeniorCitizen':    np.random.choice([0, 1], n, p=[0.84, 0.16]),
    'Churn':            np.random.choice(['No', 'Yes'], n, p=[0.73, 0.27]),
})

print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

## Step 2 — Stage 1: Profile the Dataset

In [ ]:
profile = profile_dataset(df)

print('=== Dataset Profile ===')
print(f"Shape       : {profile['shape']['rows']} rows × {profile['shape']['cols']} cols")
print(f"Missing vals: {profile['missing']}")
print(f"Skewed feats: {profile['skewed_features']}")
print(f"Corr pairs  : {len(profile['correlated_pairs'])} high-correlation pairs")
print(f"Drift flag  : {profile['drift_flag']}")
print(f"Temporal cols: {profile['temporal_columns']}")

In [ ]:
# Visualise missing values
if profile['missing']:
    miss_df = pd.DataFrame(list(profile['missing'].items()), columns=['Column', 'Missing %'])
    fig = px.bar(miss_df, x='Column', y='Missing %',
                 title='Missing Values per Column',
                 color_discrete_sequence=['#CC0000'],
                 template='simple_white')
    fig.show()
else:
    print('No missing values. Dataset is clean.')

In [ ]:
# Visualise class distribution
fig2 = px.histogram(df, x='Churn', color='Churn',
                    title='Target Class Distribution',
                    color_discrete_map={'No': '#1A1A1A', 'Yes': '#CC0000'},
                    template='simple_white')
fig2.show()

## Step 3 — Stage 2: Classify Task (with Devil's Advocate)

In [ ]:
problem_text = 'Predict which customers will churn based on their service usage and contract details'
target_col   = 'Churn'

task_result = classify_task(df, profile, problem_text, target_col)

print('=== Task Classification ===')
print(f"Task       : {task_result['task']}")
print(f"Confidence : {task_result['confidence']}")
print(f"Metric     : {task_result['metric']}")
if task_result['ontology_match']:
    print(f"ABB Ontology: {task_result['ontology_match']}")
print()
print('Reasoning chain:')
for r in task_result['reasoning']:
    print(f'  + {r}')
print()
print("Devil's Advocate check:")
for d in task_result['devil_advocate']:
    print(f'  ? {d}')
print()
print('Preprocessing plan:')
for p in task_result['preprocessing_plan']:
    print(f'  - {p}')

## Step 4 — Stage 3: AutoML

In [ ]:
automl_result = run_automl(df, task_result, target_col, budget='Fast')

if automl_result['error']:
    print(f'AutoML error: {automl_result["error"]}')
else:
    print(f'Best model : {automl_result["best_model_name"]}')
    print(f'Metrics    : {automl_result["metrics"]}')

In [ ]:
# Show leaderboard
if automl_result['leaderboard'] is not None:
    print('=== Model Leaderboard ===')
    display(automl_result['leaderboard'])

## Step 5 — Stage 4: Feature Importance

In [ ]:
feature_imp = get_feature_importance(df, target_col, task_result['task'])

if not feature_imp.empty and 'Error' not in feature_imp.columns:
    print('Top 10 features:')
    print(feature_imp.head(10).to_string(index=False))

    fig3 = px.bar(
        feature_imp.head(10),
        x='Importance', y='Feature',
        orientation='h',
        color='Importance',
        color_continuous_scale=['#f5f5f5', '#CC0000'],
        title='Top 10 Features by Importance',
        template='simple_white'
    )
    fig3.update_layout(yaxis=dict(autorange='reversed'), coloraxis_showscale=False)
    fig3.show()
else:
    print('Feature importance could not be computed:', feature_imp)

## Step 6 — Stage 5: Tri-Persona Report

In [ ]:
report = generate_report(
    profile, task_result, automl_result,
    feature_imp, problem_text,
    consequence='false_negative_costly'
)

eng = report['engineer']
mgr = report['manager']
reg = report['regulator']

# ── Engineer Mode ──
print('=' * 60)
print('ENGINEER MODE — Technical Report')
print('=' * 60)
print(f"Model         : {eng['model']}")
print(f"Task          : {eng['task']}")
print(f"Confidence    : {eng['confidence']}")
print(f"Top features  : {eng['top_features']}")
print(f"Drift flag    : {eng['drift_flag']}")
print(f"Metrics       : {eng['metrics']}")
print()

In [ ]:
# ── Manager Mode ──
print('=' * 60)
print('MANAGER MODE — Business Summary')
print('=' * 60)
print(f"Finding       : {mgr['finding']}")
print(f"Key metric    : {mgr['metric']}")
print(f"Top drivers   : {mgr['top_drivers']}")
print(f"Risk level    : {mgr['risk']}")
print(f"Confidence    : {mgr['confidence']}")
print(f"Action        : {mgr['action']}")
print(f"Next check    : {mgr['next_check']}")
print()

In [ ]:
# ── Regulator Mode ──
print('=' * 60)
print('REGULATOR MODE — Model Card')
print('=' * 60)
print(f"Model         : {reg['model']}")
print(f"Training date : {reg['training_date']}")
print(f"Training rows : {reg['training_rows']}")
print(f"Metrics       : {reg['metrics']}")
print(f"Bias check    : {reg['bias_check']}")
print(f"Drift status  : {reg['drift_status']}")
print(f"Recommendation: {reg['recommendation']}")

## Summary

This notebook demonstrated the full ARIA pipeline:

| Stage | What happened |
|---|---|
| Profile | Dataset shape, missing values, skew, correlations, drift pre-scan |
| Classify | Task type detected with Devil's Advocate check + ABB ontology match |
| AutoML | Multiple models compared, best selected, metrics computed |
| Explain | Feature importance ranked by Random Forest |
| Report | Three persona-specific outputs generated simultaneously |

**Try it with your own dataset:**
Replace the `df = pd.DataFrame(...)` cell with `df = pd.read_csv('your_file.csv')` and update `target_col` and `problem_text`.